# File 00 — Data Cleaning (REVISI)
**EHSL-ML+ | Disertasi Nanan Soekarna**

**Revisi dari versi sebelumnya.** Versi pertama keliru: 1 responden hanya dikaitkan ke 1 aktor. Proposal (§3.5.7) menegaskan tiap narapidana bisa menilai **beberapa aktor sekaligus** (investigator, prosecutor, judge, advocate, correctional officer) — hanya aktor yang benar-benar mereka temui selama proses hukum. Versi ini memperbaiki struktur tersebut.

## Struktur data (2 tabel)

| Tabel | Isi | Level |
|---|---|---|
| `df_respondent` | Demografi (usia, gender, pendidikan, jenis kasus, lama vonis, masa tahanan dijalani, lapas, wilayah) + item **Governance/Rehabilitation (GR)** — komponen ke-3 NREI | 1 baris = 1 narapidana |
| `df_actor` | Item **VfV, FCNC, ID** — dijawab terpisah untuk tiap aktor yang ditemui | 1 baris = 1 kombinasi narapidana x aktor |

Kenapa dipisah begini (bukan 1 tabel besar)? Karena GR sifatnya soal pengalaman institusi/pembinaan secara umum (ditanya sekali), sedangkan VfV/FCNC/ID soal perlakuan aktor spesifik (diulang per aktor yang ditemui). Menggabungkan keduanya dalam 1 tabel akan menduplikasi data GR sebanyak jumlah aktor yang ditemui tiap orang — boros dan berisiko salah hitung saat agregasi nanti.

**Catatan penting:** probabilitas seorang narapidana "menemui" tiap aktor (Investigator & CorrectionalOfficer 100%, Prosecutor 92%, Judge 90%, Advocate 65%) adalah **asumsi simulasi saya**, bukan angka dari proposal — proposal tidak memberi angka pasti karena itu tergantung data lapangan riil nanti.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

## 1. Bangkitkan Data Dummy — Tabel Responden

Demografi + 5 item GR (1 di antaranya reverse-coded: GR3, sesuai pola di proposal — kalimat negatif soal akses pengaduan).

In [2]:
actor_categories = ["Investigator", "Prosecutor", "Judge", "Advocate", "CorrectionalOfficer"]
regions = ["Jawa Barat", "Jawa Timur", "Sumatera Utara", "Sulawesi Selatan", "Kalimantan Timur"]
case_types = ["Narkotika", "Pencurian", "Penipuan", "Kekerasan", "Korupsi"]
institutions_by_region = {
    "Jawa Barat": ["Lapas Sukamiskin", "Lapas Cirebon"],
    "Jawa Timur": ["Lapas Porong", "Lapas Madiun"],
    "Sumatera Utara": ["Lapas Tanjung Gusta"],
    "Sulawesi Selatan": ["Lapas Makassar"],
    "Kalimantan Timur": ["Lapas Samarinda"],
}
item_groups = {
    "VfV":  ["VfV1", "VfV2", "VfV3", "VfV4", "VfV5"],
    "FCNC": ["FCNC1", "FCNC2", "FCNC3", "FCNC4", "FCNC5"],
    "ID":   ["ID1", "ID2", "ID3", "ID4", "ID5"],
}
gr_items = ["GR1", "GR2", "GR3", "GR4", "GR5"]
reverse_items_actor = ["VfV5", "FCNC3", "FCNC4", "ID3"]
reverse_items_gr = ["GR3"]
all_actor_items = [it for grp in item_groups.values() for it in grp]

encounter_prob = {  # ASUMSI simulasi, bukan dari proposal
    "Investigator": 1.00, "CorrectionalOfficer": 1.00,
    "Prosecutor": 0.92, "Judge": 0.90, "Advocate": 0.65,
}

N_RESPONDENTS = 200
resp_rows = []
for i in range(N_RESPONDENTS):
    resp_id = f"R{i+1:04d}"
    region = np.random.choice(regions)
    institution = np.random.choice(institutions_by_region[region])
    sentence_len = int(np.clip(np.random.exponential(24), 3, 240))
    time_served = int(np.clip(sentence_len * np.random.uniform(0.1, 0.95), 1, sentence_len))
    gr_base = np.random.normal(3.0, 0.7)
    row = {
        "respondent_id": resp_id,
        "age": int(np.clip(np.random.normal(34, 9), 19, 65)),
        "gender": np.random.choice(["Laki-laki", "Perempuan"], p=[0.92, 0.08]),
        "education": np.random.choice(["SD", "SMP", "SMA", "D3/S1"], p=[0.2, 0.3, 0.35, 0.15]),
        "case_type": np.random.choice(case_types),
        "sentence_length_months": sentence_len,
        "time_served_months": time_served,
        "institution": institution,
        "region": region,
    }
    for it in gr_items:
        if it in reverse_items_gr:
            val = (6 - gr_base) + np.random.normal(0, 0.9)
        else:
            val = gr_base + np.random.normal(0, 0.9)
        row[it] = int(np.clip(round(val), 1, 5))
    resp_rows.append(row)

df_respondent = pd.DataFrame(resp_rows)
print(f"df_respondent: {df_respondent.shape[0]} baris x {df_respondent.shape[1]} kolom")
df_respondent.head(3)

df_respondent: 200 baris x 14 kolom


,respondent_id,age,gender,education,case_type,sentence_length_months,time_served_months,institution,region,GR1,GR2,GR3,GR4,GR5
0,R0001,36,Laki-laki,SD,Korupsi,72,51,Lapas Makassar,Sulawesi Selatan,4,3,3,3,2
1,R0002,34,Laki-laki,SD,Kekerasan,22,2,Lapas Makassar,Sulawesi Selatan,2,2,1,2,3
2,R0003,37,Laki-laki,D3/S1,Kekerasan,21,2,Lapas Makassar,Sulawesi Selatan,1,3,5,1,3


## 2. Bangkitkan Data Dummy — Tabel Aktor (Long Format)

Untuk tiap responden, tiap aktor "dilempar koin" sesuai probabilitas encounter di atas. Kalau "kena" (ditemui), baru dibuatkan 1 baris dengan 15 item VfV/FCNC/ID. Kalau tidak ditemui, **tidak ada baris sama sekali** untuk kombinasi itu — ini otomatis merepresentasikan "not applicable" tanpa perlu placeholder eksplisit.

In [3]:
actor_rows = []
for _, r in df_respondent.iterrows():
    resp_id = r["respondent_id"]
    resp_base = np.random.normal(3.0, 0.6)
    for actor in actor_categories:
        if np.random.random() <= encounter_prob[actor]:
            actor_common = resp_base + np.random.normal(0, 0.5)
            # tiap konstruk (VfV/FCNC/ID) punya trait TERSENDIRI, masih berkorelasi lewat actor_common
            # -> supaya EFA nanti bisa menemukan 3 faktor terpisah, bukan 1 faktor besar
            construct_trait = {
                "VfV":  actor_common + np.random.normal(0, 0.55),
                "FCNC": actor_common + np.random.normal(0, 0.55),
                "ID":   actor_common + np.random.normal(0, 0.55),
            }
            row = {"respondent_id": resp_id, "actor_category": actor}
            for construct, items in item_groups.items():
                trait = construct_trait[construct]
                for it in items:
                    if it in reverse_items_actor:
                        # item negatif: simulasikan respons mentah TERBALIK dari trait sebenarnya,
                        # supaya nanti setelah di-reverse-code (6-x), arahnya kembali align
                        val = (6 - trait) + np.random.normal(0, 0.5)
                    else:
                        val = trait + np.random.normal(0, 0.5)
                    row[it] = int(np.clip(round(val), 1, 5))
            actor_rows.append(row)

df_actor = pd.DataFrame(actor_rows)
print(f"df_actor: {df_actor.shape[0]} baris (dari {N_RESPONDENTS} responden)")
print(f"Rata-rata aktor ditemui per responden: {len(df_actor)/N_RESPONDENTS:.2f} dari maksimal 5")
df_actor.head(6)

df_actor: 896 baris (dari 200 responden)
Rata-rata aktor ditemui per responden: 4.48 dari maksimal 5


,respondent_id,actor_category,VfV1,VfV2,VfV3,VfV4,VfV5,FCNC1,FCNC2,FCNC3,FCNC4,FCNC5,ID1,ID2,ID3,ID4,ID5
0,R0001,Investigator,3,3,4,3,3,3,3,4,3,2,2,3,4,3,4
1,R0001,Judge,2,2,2,2,4,3,3,3,3,3,3,3,3,3,2
2,R0001,Advocate,3,3,3,4,4,3,3,2,2,4,3,4,2,3,4
3,R0001,CorrectionalOfficer,2,2,2,2,5,3,2,3,4,2,3,3,3,3,3
4,R0002,Investigator,2,2,3,3,2,3,3,2,3,3,4,4,2,4,5
5,R0002,Prosecutor,4,3,3,4,3,3,3,4,3,3,4,4,2,3,4


### Sengaja "mengotori" data untuk latihan cleaning
Sama seperti sebelumnya: duplikat baris, missing value acak, straight-lining, salah entri.

In [4]:
dup_rows = df_actor.sample(5, random_state=1).copy()
df_actor = pd.concat([df_actor, dup_rows], ignore_index=True)

rng = np.random.default_rng(7)
for it in all_actor_items:
    df_actor.loc[rng.random(len(df_actor)) < 0.03, it] = np.nan
for it in gr_items:
    df_respondent.loc[rng.random(len(df_respondent)) < 0.03, it] = np.nan

straight_idx = df_actor.sample(3, random_state=2).index
for idx in straight_idx:
    df_actor.loc[idx, all_actor_items] = np.random.choice([1, 3, 5])

err_idx = df_actor.sample(3, random_state=3).index
for j, idx in enumerate(err_idx):
    df_actor.loc[idx, all_actor_items[j]] = np.random.choice([0, 6, -1])
df_respondent.loc[df_respondent.sample(2, random_state=4).index, "age"] = [130, -5]

print(f"Data mentah siap: df_respondent={len(df_respondent)} baris, df_actor={len(df_actor)} baris")

Data mentah siap: df_respondent=200 baris, df_actor=901 baris


**Catatan teknis:** perhatikan di generator data mentah di atas, item reverse-coded (`VfV5`, `FCNC3`, `FCNC4`, `ID3`, `GR3`) dibangkitkan pakai `(6 - trait)`, bukan `trait` langsung seperti item lain. Ini disengaja: kalau item mentahnya dibuat searah dengan item lain (seolah 5 selalu berarti bagus), lalu di Langkah 3 di bawah kita balik lagi dengan reverse coding (`6-x`), hasilnya malah jadi terbalik dan bertentangan arah dengan item lain — ini pernah kejadian dan ketahuan saat Cronbach's alpha-nya keluar negatif. Jadi di data mentah, kita sengaja simulasikan seolah orang itu menjawab pertanyaan versi negatif, baru reverse coding di bawah ini meluruskannya kembali ke arah yang benar.

## 3. Reverse Coding

In [5]:
df_actor_c = df_actor.copy()
for it in reverse_items_actor:
    df_actor_c[it] = 6 - df_actor_c[it]

df_resp_c = df_respondent.copy()
for it in reverse_items_gr:
    df_resp_c[it] = 6 - df_resp_c[it]
print("Reverse coding diterapkan pada item aktor dan GR.")

Reverse coding diterapkan pada item aktor dan GR.


## 4. Cek & Hapus Duplikat

Untuk `df_actor`, duplikat dicek berdasarkan **kombinasi** `respondent_id` + `actor_category` (bukan `respondent_id` saja) — karena satu orang memang boleh muncul berkali-kali (beda aktor), yang tidak boleh adalah kombinasi yang sama persis muncul dua kali.

In [6]:
n_before = len(df_actor_c)
dupe_mask = df_actor_c.duplicated(subset=["respondent_id", "actor_category"], keep="first")
df_actor_c = df_actor_c[~dupe_mask].reset_index(drop=True)
print(f"[df_actor] Duplikat dihapus: {dupe_mask.sum()} baris ({n_before} -> {len(df_actor_c)})")

n_before_r = len(df_resp_c)
dupe_mask_r = df_resp_c.duplicated(subset=["respondent_id"], keep="first")
df_resp_c = df_resp_c[~dupe_mask_r].reset_index(drop=True)
print(f"[df_respondent] Duplikat dihapus: {dupe_mask_r.sum()} baris")

[df_actor] Duplikat dihapus: 5 baris (901 -> 896)
[df_respondent] Duplikat dihapus: 0 baris


## 5. Koreksi Nilai di Luar Skala

In [7]:
invalid_count = 0
for it in all_actor_items:
    m = ~df_actor_c[it].isna() & ~df_actor_c[it].between(1, 5)
    invalid_count += m.sum()
    df_actor_c.loc[m, it] = np.nan
age_invalid = ~df_resp_c["age"].between(17, 90)
df_resp_c.loc[age_invalid, "age"] = np.nan
print(f"[df_actor] Nilai di luar skala 1-5: {invalid_count} sel -> missing")
print(f"[df_respondent] Usia tidak masuk akal: {age_invalid.sum()} baris -> missing")

[df_actor] Nilai di luar skala 1-5: 3 sel -> missing
[df_respondent] Usia tidak masuk akal: 2 baris -> missing


## 6. Missing Value — Mengikuti Aturan Proposal §3.8.4 Persis

Ini bagian yang paling berbeda dari versi pertama. Proposal bilang:
> *"missing values may be replaced using **the respondent's mean score for the same construct**"* — bukan rata-rata semua orang, tapi rata-rata item lain **milik baris itu sendiri** dalam konstruk yang sama.
> *"If too many items are missing within a construct, the construct score for that respondent is not calculated"* — bukan dipaksa diisi, tapi dibiarkan kosong (tidak dihitung).
> *"If a questionnaire has excessive missing data, the response may be excluded"* — kalau kosongnya kebangetan di satu baris, baris itu dibuang total.

Jadi ada 3 tingkat, bukan cuma "isi rata-rata" seperti versi lama:
1. Missing sedikit dalam 1 konstruk (≤1 dari 5 item) → isi pakai rata-rata item lain **di baris yang sama**
2. Missing banyak dalam 1 konstruk (>1 dari 5 item) → skor konstruk itu untuk baris ini = kosong (tidak dipaksa dihitung)
3. Missing berlebihan di seluruh baris (>30% dari total item) → baris dibuang

In [8]:
def impute_row_construct(row, items, small_missing_max=1):
    vals = row[items]
    n_missing = vals.isna().sum()
    if n_missing == 0:
        return vals.mean()
    elif n_missing <= small_missing_max:
        return vals.mean(skipna=True)
    else:
        return np.nan

overall_missing_pct = df_actor_c[all_actor_items].isna().mean(axis=1)
excessive_mask = overall_missing_pct > 0.30
df_actor_c = df_actor_c[~excessive_mask].reset_index(drop=True)
print(f"[df_actor] Baris dibuang (missing >30% total item): {excessive_mask.sum()}")

for construct, items in item_groups.items():
    df_actor_c[f"{construct}_score"] = df_actor_c.apply(
        lambda r: impute_row_construct(r, items, small_missing_max=1), axis=1)
n_unscored = df_actor_c[[f"{c}_score" for c in item_groups]].isna().sum().sum()
print(f"[df_actor] Skor konstruk tidak dihitung (missing terlalu banyak): {n_unscored} sel")

gr_missing_pct = df_resp_c[gr_items].isna().mean(axis=1)
excessive_gr = gr_missing_pct > 0.30
df_resp_c = df_resp_c[~excessive_gr].reset_index(drop=True)
df_resp_c["GR_score"] = df_resp_c.apply(lambda r: impute_row_construct(r, gr_items, small_missing_max=1), axis=1)
print(f"[df_respondent] Baris dibuang (missing GR >30%): {excessive_gr.sum()}")
print(f"[df_respondent] Skor GR tidak dihitung: {df_resp_c['GR_score'].isna().sum()}")

[df_actor] Baris dibuang (missing >30% total item): 0


[df_actor] Skor konstruk tidak dihitung (missing terlalu banyak): 14 sel
[df_respondent] Baris dibuang (missing GR >30%): 1
[df_respondent] Skor GR tidak dihitung: 0


## 7. Straight-Lining & Outlier Statistik
Sama seperti sebelumnya — deteksi & tandai, tidak otomatis dibuang.

In [9]:
item_std = df_actor_c[all_actor_items].std(axis=1)
df_actor_c["flag_straightline"] = item_std < 0.3
print(f"Straight-lining ditandai: {df_actor_c['flag_straightline'].sum()}")

df_actor_c["total_score"] = df_actor_c[all_actor_items].mean(axis=1)
z = np.abs(stats.zscore(df_actor_c["total_score"].fillna(df_actor_c["total_score"].mean())))
df_actor_c["flag_statistical_outlier"] = z > 3
print(f"Outlier statistik ditandai: {df_actor_c['flag_statistical_outlier'].sum()}")

Straight-lining ditandai: 15
Outlier statistik ditandai: 0


## 8. Cek Distribusi

In [10]:
print("Distribusi skor per konstruk:")
for construct in item_groups:
    s = df_actor_c[f"{construct}_score"].dropna()
    print(f"{construct:5s} | mean={s.mean():.2f}  sd={s.std():.2f}  skew={stats.skew(s):.2f}  kurtosis={stats.kurtosis(s):.2f}")
s_gr = df_resp_c["GR_score"].dropna()
print(f"{'GR':5s} | mean={s_gr.mean():.2f}  sd={s_gr.std():.2f}  skew={stats.skew(s_gr):.2f}  kurtosis={stats.kurtosis(s_gr):.2f}")

Distribusi skor per konstruk:
VfV   | mean=2.95  sd=0.94  skew=0.07  kurtosis=-0.58
FCNC  | mean=2.97  sd=0.94  skew=0.08  kurtosis=-0.49
ID    | mean=2.98  sd=0.94  skew=0.06  kurtosis=-0.53
GR    | mean=2.99  sd=0.77  skew=0.09  kurtosis=-0.56


## 9. Simpan Output

Dua file output: `data_actor_clean.csv` (level responden-aktor) dan `data_respondent_clean.csv` (level responden). Keduanya jadi input File 01 (Validitas & Reliabilitas), dihubungkan lewat `respondent_id`.

In [11]:
df_actor_c.to_csv("data_actor_clean.csv", index=False)
df_resp_c.to_csv("data_respondent_clean.csv", index=False)
print(f"SELESAI.")
print(f"data_actor_clean.csv      : {len(df_actor_c)} baris (level responden x aktor)")
print(f"data_respondent_clean.csv : {len(df_resp_c)} baris (level responden)")

SELESAI.
data_actor_clean.csv      : 896 baris (level responden x aktor)
data_respondent_clean.csv : 199 baris (level responden)


---
## Ringkasan Revisi File 00

| | Versi 1 (keliru) | Versi 2 (revisi, sesuai proposal) |
|---|---|---|
| Struktur | 1 responden = 1 aktor | 1 responden = 1–5 aktor (long format) |
| GR | dicampur dengan item aktor | tabel terpisah, level responden |
| Missing value | rata-rata kolom (lintas semua orang) | rata-rata milik baris itu sendiri, per konstruk, dengan ambang batas jelas |
| Demografi | age, education, sentence_length, case_type, region | + gender, time_served_months, institution |

File berikutnya (**File 01 — Validitas & Reliabilitas**) akan pakai `data_actor_clean.csv` untuk Cronbach's alpha VfV/FCNC/ID, dan `data_respondent_clean.csv` untuk Cronbach's alpha GR.